In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch
from tqdm.auto import tqdm

In [2]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)


LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['hparas']['batch_size'] = 64
config['hparas']['global_batch_size'] = 64
config['num_gpus'] = 1 

model = LitAudioSSL(config)


In [3]:
from lightning_scripts import jsinV3DataLoader_precombined_batched 

importlib.reload(jsinV3DataLoader_precombined_batched)
MatchedSpeechInNoiseDatasetBatched = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched

dataset = MatchedSpeechInNoiseDatasetBatched(speech_h5_path=config['data']['val_speech_h5_path'],
                                                     noise_h5_path=config['data']['val_noise_h5_path'],
                                                     low_db=config['audio_transforms']['low_snr'],
                                                     high_db=config['audio_transforms']['high_snr'],
                                                     db_spl=config['audio_transforms']['dbspl'],
                                                     batch_size=config['hparas']['batch_size'],
                                                     signal_augment=config['data'].get("signal_augment", False),
                                                     target_keys=config['data'].get("target_keys", None),
                                                     )
dataset[0]

([tensor([[ 2.9755e-02,  3.0070e-02,  1.4406e-02,  ..., -1.4620e-02,
           -1.8493e-02, -1.9111e-02],
          [ 7.0083e-03,  8.9769e-03,  1.2000e-02,  ..., -2.8257e-03,
           -3.5366e-03, -3.2285e-03],
          [-3.1491e-03, -3.8565e-03, -5.0330e-04,  ..., -2.6268e-02,
           -2.6733e-02, -2.2242e-02],
          ...,
          [-1.0390e-02, -1.8595e-02,  1.9440e-02,  ..., -1.3699e-03,
           -1.0850e-03, -8.7618e-04],
          [ 8.3191e-03,  5.7712e-03,  1.0358e-02,  ..., -4.8824e-03,
           -7.3218e-03, -1.0071e-02],
          [-8.5021e-05, -3.5469e-03, -2.1388e-03,  ..., -7.9756e-03,
           -8.2620e-03, -9.2424e-03]]),
  tensor([[-3.4793e-03, -4.4580e-03, -4.4105e-03,  ...,  1.2766e-02,
            1.3590e-02,  1.4466e-02],
          [-1.9640e-03, -1.1352e-02,  1.2533e-03,  ...,  3.2192e-04,
            1.7198e-04,  5.3196e-05],
          [ 2.5090e-03,  1.6021e-02,  2.1391e-02,  ...,  1.3196e-02,
           -1.2344e-03, -7.4797e-03],
          ...,
     

In [4]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA H100 PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose pa

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 317
This is the first step after restoring from a checkpoint!


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined